# Data Engineering
---

## 1 - Data Loading

In [122]:
import json
from pymongo import MongoClient

# connect to MongoDB and initialize client 
client = MongoClient('mongodb://localhost:27017/')
db = client['novacred_db']
collection = db['applications'] # define collection for more reliability  

# drop collection before loading multiple times
collection.drop()

# load JSON file
with open('../data/raw_credit_applications.json', 'r') as file:
    raw_data = json.load(file)

## 2 - Data Cleaning 

Following the project guidelines, we will address the six core data quality issues:
### 1. Duplicate records
MongoDB requires unique _id fields. Therefore, we check for duplicates.

In [123]:
unique_records = {}
duplicates_count = 0
# as "_id" is a unique identifier, documents with the same _id shall be deleted 
for application in raw_data:
    app_id = application['_id']
    
    # count duplicates
    if app_id in unique_records:
        duplicates_count += 1
        print(f"Duplicate found: {app_id}")
    
    # duplicates will be removed through overwriting
    unique_records[app_id] = application

clean_data = list(unique_records.values())

print(f"Found and removed {duplicates_count} duplicates.")

Duplicate found: app_042
Duplicate found: app_001
Found and removed 2 duplicates.


The duplicated documents are resubmitted applications with duplicate IDs. We will kept the most recent resubmission and load the unique records into the database.

In [124]:
# loading cleaned data 
collection.insert_many(clean_data)
print(f"Inserted {len(clean_data)} unique documents into MongoDB.")

Inserted 500 unique documents into MongoDB.


### 2. Inconsistent data types
Instead of guessing which fields have inconsistent data types, we will dynamically audit the entire collection. 
This script flattens every document and records the Python data types found for each field. If a field contains more than one data type across the dataset (e.g., both integers and strings), it will be flagged for cleaning.

In [125]:
from collections import defaultdict

# Dictionary to hold sets of data types for every field
field_types = defaultdict(set)

# Helper function to flatten nested JSON (e.g., turns {'financials': {'income': 50}} into 'financials.income': 50)
def flatten_doc(d, parent_key='', sep='.'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_doc(v, new_key, sep=sep).items())
        elif isinstance(v, list):
            # For this audit, we skip complex arrays like spending_behavior
            pass 
        else:
            # We ignore null/None values for the type consistency check
            if v is not None and v != "":
                items.append((new_key, type(v).__name__))
    return dict(items)

# Scan every document in the database
for doc in collection.find({}):
    doc.pop('_id', None) # We don't need to check the MongoDB ID
    flat_doc = flatten_doc(doc)
    
    for field, data_type in flat_doc.items():
        field_types[field].add(data_type)

inconsistent_fields = {field: types for field, types in field_types.items() if len(types) > 1}

if not inconsistent_fields:
    print("No inconsistent data types found.")
else:
    for field, types in inconsistent_fields.items():
        print(f"FLAGGED: '{field}' contains mixed types: {types}")

FLAGGED: 'financials.annual_income' contains mixed types: {'str', 'float', 'int'}


In [126]:
# audit flagged 'financials.annual_income' as having mixed string/int types
# We will dynamically find any strings in these numeric fields and cast them to integers

fields_to_fix = list(inconsistent_fields.keys())

for field in fields_to_fix:
    # Update query: Find documents where this field is a string, and cast it to an int
    result = collection.update_many(
        {field: {"$type": "string"}},
        [{"$set": {field: {"$toInt": f"${field}"}}}]
    )
    if result.modified_count > 0:
        print(f"Fixed: Casted {result.modified_count} string values to integers in '{field}'.")

Fixed: Casted 8 string values to integers in 'financials.annual_income'.


### 3. Missing or incomplete records
Before altering the database, we must audit **all** fields for missing values (nulls, empty strings, or completely missing keys). 

**Business Rules for Handling Missing Data:**
When missing data is found, we apply the following logic based on the field's importance to the credit application:
1. **Critical Identifiers (e.g., SSN):** A credit application is legally invalid without an SSN. If this is missing, the entire record must be **dropped**.
2. **Contact Information (e.g., Email):** Non-critical for the credit risk algorithm. If missing, we will **impute** it with a placeholder (e.g., "UNKNOWN") to preserve the rest of the valid financial data.
3. **Financial/Algorithmic Data:** If critical numeric fields used for bias analysis (like income) are missing, they must be evaluated for dropping or median imputation.

In [127]:
# 1. First, we dynamically find EVERY possible field name in our dataset
all_fields = set()
for doc in collection.find({}):
    doc.pop('_id', None) # Ignore the MongoDB ID
    # We reuse the flatten_doc function from Step 2 to get all nested keys
    all_fields.update(flatten_doc(doc).keys())

# 2. Now, we query MongoDB to count missing values for every single field
missing_report = {}

for field in all_fields:
    missing_count = collection.count_documents({
        "$or": [
            {field: {"$exists": False}},
            {field: None},
            {field: ""}
        ]
    })
    if missing_count > 0:
        missing_report[field] = missing_count

print("--- COMPREHENSIVE MISSING DATA AUDIT ---")
if not missing_report:
    print("No missing data found in any field.")
else:
    for field, count in sorted(missing_report.items()):
        print(f"FLAGGED: '{field}' is missing or empty in {count} records.")

--- COMPREHENSIVE MISSING DATA AUDIT ---
FLAGGED: 'applicant_info.date_of_birth' is missing or empty in 5 records.
FLAGGED: 'applicant_info.email' is missing or empty in 7 records.
FLAGGED: 'applicant_info.gender' is missing or empty in 3 records.
FLAGGED: 'applicant_info.ip_address' is missing or empty in 5 records.
FLAGGED: 'applicant_info.ssn' is missing or empty in 5 records.
FLAGGED: 'applicant_info.zip_code' is missing or empty in 2 records.
FLAGGED: 'decision.approved_amount' is missing or empty in 208 records.
FLAGGED: 'decision.interest_rate' is missing or empty in 208 records.
FLAGGED: 'decision.rejection_reason' is missing or empty in 292 records.
FLAGGED: 'financials.annual_income' is missing or empty in 5 records.
FLAGGED: 'financials.annual_salary' is missing or empty in 495 records.
FLAGGED: 'loan_purpose' is missing or empty in 450 records.
FLAGGED: 'notes' is missing or empty in 498 records.
FLAGGED: 'processing_timestamp' is missing or empty in 438 records.


**Executing the Missing Data Fixes:**
Based on our comprehensive audit, we have discovered patterns in the missing data. We will apply the following business rules:
1. **Drop:** Records missing `ssn` are structurally invalid.
2. **Impute:** Categorical and demographic fields (`gender`, `email`, `zip_code`) will be filled with an "UNKNOWN" placeholder.
3. **Keep as Null:** Conditional fields (like `approved_amount` vs `rejection_reason`) and sparse fields (like `notes`) will be left alone. These will later naturally convert to `NaN` in Pandas.

In [128]:
# 1. DROP the 5 records missing an SSN
deleted = collection.delete_many({
    "$or": [{"applicant_info.ssn": {"$exists": False}}, {"applicant_info.ssn": None}, {"applicant_info.ssn": ""}]
})
print(f"Dropped {deleted.deleted_count} completely invalid records (Missing SSN).")

# 2. IMPUTE "UNKNOWN" for demographics and contact info
# Fix Emails
updated_emails = collection.update_many(
    {"$or": [{"applicant_info.email": {"$exists": False}}, {"applicant_info.email": None}, {"applicant_info.email": ""}]},
    {"$set": {"applicant_info.email": "UNKNOWN_EMAIL"}}
)
print(f"Imputed {updated_emails.modified_count} missing emails.")

# Fix Gender (Crucial for Bias Analysis)
updated_genders = collection.update_many(
    {"$or": [{"applicant_info.gender": {"$exists": False}}, {"applicant_info.gender": None}, {"applicant_info.gender": ""}]},
    {"$set": {"applicant_info.gender": "UNKNOWN"}}
)
print(f"Imputed {updated_genders.modified_count} missing genders.")

# Fix Zip Codes (Crucial for Redlining Analysis)
updated_zips = collection.update_many(
    {"$or": [{"applicant_info.zip_code": {"$exists": False}}, {"applicant_info.zip_code": None}, {"applicant_info.zip_code": ""}]},
    {"$set": {"applicant_info.zip_code": "UNKNOWN_ZIP"}}
)
print(f"Imputed {updated_zips.modified_count} missing zip codes.")

Dropped 5 completely invalid records (Missing SSN).
Imputed 3 missing emails.
Imputed 0 missing genders.
Imputed 0 missing zip codes.


Based on the audit results, the 5 records missing an SSN were completely blank "ghost" applications that also accounted for the missing demographic fields (such as gender and zip code). Therefore, dropping these 5 structurally invalid applications simultaneously resolved the missing data issues across those other fields, leaving a clean dataset for the Data Scientist.

### 4. Inconsistent coding/formatting of categorical fields
To ensure our categorical variables are formatted consistently, we first audit the database to extract every unique value present in these fields. Once we identify the inconsistent variations (e.g., abbreviations, varying capitalization), we will apply a strict mapping logic to standardize them.

In [129]:
# Define the known categorical fields
categorical_fields = [
    "applicant_info.gender", 
    "loan_purpose", 
    "decision.rejection_reason"
]

for field in categorical_fields:
    # Use MongoDB's distinct() to grab all unique values currently in the database
    unique_values = collection.distinct(field)
    
    # Filter out None/UNKNOWN just to see the actual messy data
    clean_unique_values = [v for v in unique_values if v not in [None, "UNKNOWN"]]
    
    print(f"\nUnique values found in '{field}':")
    print(clean_unique_values)


Unique values found in 'applicant_info.gender':
['F', 'Female', 'M', 'Male']

Unique values found in 'loan_purpose':
['auto', 'business', 'debt_consolidation', 'education', 'home_improvement', 'medical', 'moving', 'personal', 'vacation', 'wedding']

Unique values found in 'decision.rejection_reason':
['algorithm_risk_score', 'high_dti_ratio', 'insufficient_credit_history', 'low_income']


**Executing the Categorical Standardization:**
Based on the audit results, the `applicant_info.gender` field contains inconsistent coding (mixing "M"/"Male" and "F"/"Female"). We will standardize these to use the full words "Male" and "Female" to ensure accurate demographic groupings for the downstream Bias Analysis.

In [130]:
# We only need to target the specific abbreviations found in our audit
gender_fixes = {
    "M": "Male",
    "F": "Female"
}

total_fixed = 0

for messy_value, standard_value in gender_fixes.items():
    # Find any document with the abbreviated gender and replace it with the full word
    updated_gender = collection.update_many(
        {"applicant_info.gender": messy_value},
        {"$set": {"applicant_info.gender": standard_value}}
    )
    
    if updated_gender.modified_count > 0:
        print(f"Standardized {updated_gender.modified_count} records from '{messy_value}' to '{standard_value}'.")
        total_fixed += updated_gender.modified_count

if total_fixed == 0:
    print("No categorical formatting fixes were needed.")
else:
    print(f"\nFixed a total of {total_fixed} records.")

Standardized 53 records from 'M' to 'Male'.
Standardized 58 records from 'F' to 'Female'.

Fixed a total of 111 records.


### 5. Invalid or impossible values
In the context of a credit application, financial figures (like income or debt) and time-based figures (like credit history duration) cannot logically be less than zero. We will dynamically audit the entire database to find any numeric field that contains negative values.

In [131]:
# 1. Ensure a list of all fields. 
# (Re-using the logic from Step 2 to ensure this cell can run independently)
all_fields = set()
for doc in collection.find({}):
    doc_copy = dict(doc)
    doc_copy.pop('_id', None)
    all_fields.update(flatten_doc(doc_copy).keys())

impossible_report = {}

# 2. Dynamically scan all fields for negative numbers and store the documents
for field in all_fields:
    # Query for numbers less than 0
    query = {
        "$and": [
            {field: {"$type": ["int", "double", "long", "decimal"]}},
            {field: {"$lt": 0}}
        ]
    }
    
    # Fetch the actual broken documents
    bad_docs = list(collection.find(query))
    
    if len(bad_docs) > 0:
        impossible_report[field] = bad_docs

if not impossible_report:
    print("No negative or impossible numeric values found.")
else:
    for field, docs in sorted(impossible_report.items()):
        print(f"\nFLAGGED: '{field}' contains {len(docs)} records with negative values.")
        print("Affected documents:")
        
        for doc in docs:
            # Flatten the document just to easily print the exact nested value
            doc_copy = dict(doc)
            doc_copy.pop('_id', None)
            flat = flatten_doc(doc_copy)
            bad_value = flat.get(field)
            
            print(f"   - Application ID: {doc.get('_id')} | {field} = {bad_value}")


FLAGGED: 'financials.credit_history_months' contains 2 records with negative values.
Affected documents:
   - Application ID: app_043 | financials.credit_history_months = int
   - Application ID: app_156 | financials.credit_history_months = int

FLAGGED: 'financials.savings_balance' contains 1 records with negative values.
Affected documents:
   - Application ID: app_290 | financials.savings_balance = int


**Executing the Impossible Values Fix:**
Based on our audit, we identified 2 records with negative `credit_history_months` and 1 record with a negative `savings_balance`. 

**Business Logic for Correction:** Negative durations for credit history are physically impossible, and negative savings balances are invalid for this institution's data model. These are classic typographical errors (e.g., a stray dash interpreted as a minus sign during data entry). To preserve the data for the Bias Analysis, we will apply a mathematical absolute value transformation to correct the signs rather than dropping the records entirely.

In [132]:
# The fields our audit flagged
flagged_numeric_fields = [
    "financials.credit_history_months",
    "financials.savings_balance"
]

total_fixed = 0

for field in flagged_numeric_fields:
    # Update query: Find negative values and apply the $abs (absolute value) operator
    corrected = collection.update_many(
        {field: {"$lt": 0}},
        [{"$set": {field: {"$abs": f"${field}"}}}]
    )
    
    if corrected.modified_count > 0:
        print(f"Converted {corrected.modified_count} negative values in '{field}' to absolute values.")
        total_fixed += corrected.modified_count

print(f"\nImpossible values correction complete. Fixed a total of {total_fixed} records.")

Converted 2 negative values in 'financials.credit_history_months' to absolute values.
Converted 1 negative values in 'financials.savings_balance' to absolute values.

Impossible values correction complete. Fixed a total of 3 records.
